MAKEMORE_II: CHARACTER LANGUAGE MODEL

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline 

In [2]:
words = open('names.txt','r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
block_size = 3
X, Y = [],[]
for w in words[:5]:
    print(w)
    context = [0]*block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)  

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [6]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

Lookup Table C: (Input layer)

In [7]:
C = torch.randn((27, 2))

In [8]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

First Hidden Layer: Tanh

In [9]:
W1 = torch.randn((6,100)) #6 inputs and 100 neurons
b1 = torch.randn(100) #each neuron has a bias

In [10]:
torch.cat([emb[:,0,:], emb[:,1,:],emb[:,2,:]],1).shape #cat concatenates the first 3 embeddings

torch.Size([32, 6])

In [11]:
torch.cat(torch.unbind(emb, 1),1).shape #unbind helps us do the same thing as previous line, but without having to call emb slicing again and again.

torch.Size([32, 6])

In [12]:
emb.view(32,6).shape #View helps us view the same tensor in different dimensions. Therefore this is the same as the previous line, but more efficient as cat creates another memory space, view doesnt

torch.Size([32, 6])

In [13]:
#Therefore
h = torch.tanh(emb.view(-1,6) @ W1 + b1) #tanh layer

In [14]:
h

tensor([[-0.7694,  0.9978, -0.9998,  ...,  0.9710,  0.1522, -0.9998],
        [-0.6013,  0.9277, -0.9980,  ...,  0.9958, -0.4830, -0.9984],
        [-0.4345,  0.4746, -0.8156,  ...,  0.9968, -0.7935, -0.8875],
        ...,
        [-0.9053, -0.9994,  0.9373,  ...,  0.9999, -0.2925, -0.9181],
        [-0.5300, -0.9994,  0.9932,  ...,  0.9996, -0.9029, -0.3849],
        [-0.4299, -0.9921,  0.9723,  ...,  0.9978,  0.0828, -0.3755]])

In [15]:
h.shape

torch.Size([32, 100])

Final Layer: Softmax

In [16]:
W2 = torch.randn((100,27))
b2 = torch.randn(27)

In [17]:
logits = h @ W2 + b2

In [18]:
logits.shape

torch.Size([32, 27])

In [19]:
counts = logits.exp()

In [20]:
prob = counts/counts.sum(1,keepdims=True)

In [21]:
prob.shape

torch.Size([32, 27])

In [22]:
prob[0].sum()

tensor(1.)

In [23]:
loss = -prob[torch.arange(32), Y].log().mean()
loss #negative log likelihood

tensor(18.3333)

Neural Network Summary:

Getting Input Data and Labels:

In [24]:
X.shape, Y.shape

(torch.Size([32, 3]), torch.Size([32]))

Getting the parameters: C,W1,b1,W2,b2

In [25]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2), generator=g)
W1 = torch.randn((6,100), generator=g)
b1 = torch.randn(100,generator=g)
W2 = torch.randn((100,27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

Total number of params:

In [26]:
sum(p.nelement() for p in parameters)

3481

Hidden and output Layer: Tanh, Softmax

In [27]:
#Hidden:
emb = C[X] #(32,3,2)
h = torch.tanh(emb.view(-1,6) @ W1 + b1) #(32,100)

#Output:
logits = h @W2 +b2 #(32,27)
counts = logits.exp()
prob = counts/counts.sum(1,keepdims=True)
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(17.7697)

F.cross_entropy: (loss)

In [28]:
F.cross_entropy(logits,Y)

tensor(17.7697)

Advantages of F.cross_entropy: 1. Better forward pass 2. Better backward pass 3. No Tensor inconsistencies

Training loop:

In [29]:
for p in parameters:
    p.requires_grad=True

In [32]:
for _ in range(1000): 
    #NN:
    #Forward Pass: 
    emb = C[X] #(32,3,2)
    h = torch.tanh(emb.view(-1,6) @ W1 + b1) #(32,100)
    logits = h @W2 +b2 #(32,27)
    loss = F.cross_entropy(logits, Y)
    #Backward Pass:
    for p in parameters:
        p.grad = None
    loss.backward()

    #Update: 
    for p in parameters:
        p.data += -0.1 * p.grad
print(loss.item())

0.25560441613197327


Here params: 3481, and only 32 input examples THEREFORE Overfitting

Training on FULL Dataset:

In [34]:
block_size = 3
X, Y = [],[]
for w in words:
    #print(w)
    context = [0]*block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
       #print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)  

In [35]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [36]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2), generator=g)
W1 = torch.randn((6,100), generator=g)
b1 = torch.randn(100,generator=g)
W2 = torch.randn((100,27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [37]:
sum(p.nelement() for p in parameters)

3481

In [39]:
for p in parameters:
    p.requires_grad=True

In [41]:
for _ in range(10): 
    #NN:
    #Forward Pass: 
    emb = C[X] #(32,3,2)
    h = torch.tanh(emb.view(-1,6) @ W1 + b1) #(32,100)
    logits = h @W2 +b2 #(32,27)
    loss = F.cross_entropy(logits, Y)
    print(loss.item())
    #Backward Pass:
    for p in parameters:
        p.grad = None
    loss.backward()

    #Update: 
    for p in parameters:
        p.data += -0.1 * p.grad

10.709586143493652
10.407631874084473
10.127808570861816
9.864364624023438
9.614501953125
9.376439094543457
9.148944854736328
8.931109428405762
8.722230911254883
8.521748542785645


MiniBatches: (take a part of the dataset to make training faster)

In [43]:
for _ in range(100): 
    #MiniBatch:
    ix = torch.randint(0, X.shape[0], (32,))
    
    #NN:
    #Forward Pass: 
    emb = C[X[ix]] #(32,3,2)
    h = torch.tanh(emb.view(-1,6) @ W1 + b1) #(32,100)
    logits = h @W2 +b2 #(32,27)
    loss = F.cross_entropy(logits, Y[ix])
    print(loss.item())

    #Backward Pass:
    for p in parameters:
        p.grad = None
    loss.backward()

    #Update: 
    for p in parameters:
        p.data += -0.1 * p.grad

7.958664894104004
5.632733345031738
7.849127292633057
5.748197078704834
6.853358745574951
7.7316484451293945
6.652688026428223
6.388394355773926
7.117591857910156
6.8368072509765625
7.054397106170654
4.4592790603637695
5.778332710266113
5.371140956878662
7.456064224243164
6.123467445373535
4.962241172790527
6.071657180786133
6.677762985229492
7.42449426651001
5.991232872009277
3.8335046768188477
4.926332473754883
6.012678146362305
4.869268894195557
4.806843280792236
4.408684730529785
4.169397354125977
4.328413963317871
4.3924479484558105
3.8744304180145264
5.512571811676025
3.713040590286255
4.492799758911133
4.944676876068115
4.8668622970581055
5.498751640319824
4.6738715171813965
5.665056228637695
3.6484270095825195
4.82904577255249
3.140136480331421
4.997013568878174
4.108586311340332
3.7089383602142334
3.2746388912200928
3.6949234008789062
4.886606216430664
4.369509696960449
5.297098159790039
4.131623268127441
4.241974353790283
3.6598048210144043
4.2903032302856445
3.75542688369750

Notice how with the minibatch the output came out faster